# Download and process WHACS offshore waves

Download WHACS bulk and partition spectra, remap them onto metamodel goals, and
write the processed zarr used by dynamic forcing and hindcast reconstruction.

**Prerequisites:** ``04_Select_Goals_by_Hindcasts.ipynb`` (``<Case>_Goals.json``).

## Configuration


In [ ]:
import numpy as np
import xarray as xr

from utils.config import get_offshore_wave, load_goals, resolve

whacs_data = xr.open_dataset(
    resolve("inputs/WHACS/processed/WHACS_seapoint.zarr")
).load()
whacs_data = whacs_data.assign({"tp": 1.0 / whacs_data["fp"]})
whacs_data["tp"].attrs = {
    "long_name": "Wave peak period",
    "units": "s",
}
whacs_data

In [ ]:
# Offshore node per goal, matched by lon/lat to the goals JSON snap points
# (hindcast.whacs.longitude/latitude). Some hindcasts are gridded rather
# than point-organized, so matching by position is more robust than by id.
goals = load_goals()
seapoint_lon = whacs_data["longitude"].isel(time=-1)
seapoint_lat = whacs_data["latitude"].isel(time=-1)
ordered_seapoints_for_goals = []
for goal in goals.values():
    snap = goal["hindcast"]["whacs"]
    distance = np.hypot(
        seapoint_lon - snap["longitude"], seapoint_lat - snap["latitude"]
    )
    ordered_seapoints_for_goals.append(
        int(whacs_data["seapoint"][int(distance.argmin())])
    )

whacs_data = whacs_data.sel(seapoint=ordered_seapoints_for_goals)[
    [
        "hs",
        "tp",
        "dir",
        "spr",
        "phs0",
        "ptp0",
        "pdir0",
        "pspr0",
        "phs1",
        "ptp1",
        "pdir1",
        "pspr1",
        "phs2",
        "ptp2",
        "pdir2",
        "pspr2",
        "phs3",
        "ptp3",
        "pdir3",
        "pspr3",
    ]
].rename({"seapoint": "goal"})
whacs_data["goal"] = [int(k) for k in goals]
whacs_data = whacs_data.assign_coords(
    {"seapoint": ("goal", ordered_seapoints_for_goals)}
)
whacs_data.to_zarr(resolve(get_offshore_wave("WHACS")["data_path"]))

In [ ]:
xr.open_zarr(resolve(get_offshore_wave("WHACS")["data_path"]))